Due to unavailability of Azure Databricks access,
the ETL workflow was implemented using PySpark
in Google Colab.

In [1]:
!pip install pyspark
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder .appName("SupplyChainETL") .getOrCreate()
print("Spark Session Created")

Spark Session Created


In [2]:
data = {"order_id": [1,2,3,4,5],"supplier_id": [101,102,101,103,104],"delivery_date": ["2026-05-01","2026-05-10","2026-05-14","2026-04-28","2026-05-15"]}
df = pd.DataFrame(data)
print(df)

   order_id  supplier_id delivery_date
0         1          101    2026-05-01
1         2          102    2026-05-10
2         3          101    2026-05-14
3         4          103    2026-04-28
4         5          104    2026-05-15


In [3]:
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

In [4]:
df['delay_days'] = (pd.Timestamp.today() -df['delivery_date']).dt.days

In [6]:
df['is_delayed'] = np.where(df['delay_days'] > 0,1,0)
print(df[['order_id','supplier_id','delay_days','is_delayed']])

   order_id  supplier_id  delay_days  is_delayed
0         1          101          13           1
1         2          102           4           1
2         3          101           0           0
3         4          103          16           1
4         5          104          -1           0


In [7]:
df.to_csv("cleaned_orders.csv",index=False)
print("CSV Saved")

CSV Saved


In [8]:
orders_df = spark.read.csv("cleaned_orders.csv",header=True,inferSchema=True)
orders_df.show()

+--------+-----------+-------------+----------+----------+
|order_id|supplier_id|delivery_date|delay_days|is_delayed|
+--------+-----------+-------------+----------+----------+
|       1|        101|   2026-05-01|        13|         1|
|       2|        102|   2026-05-10|         4|         1|
|       3|        101|   2026-05-14|         0|         0|
|       4|        103|   2026-04-28|        16|         1|
|       5|        104|   2026-05-15|        -1|         0|
+--------+-----------+-------------+----------+----------+



In [9]:
from pyspark.sql.functions import col
delayed_df = orders_df.filter(col("is_delayed") == 1)
delayed_df.show()

+--------+-----------+-------------+----------+----------+
|order_id|supplier_id|delivery_date|delay_days|is_delayed|
+--------+-----------+-------------+----------+----------+
|       1|        101|   2026-05-01|        13|         1|
|       2|        102|   2026-05-10|         4|         1|
|       4|        103|   2026-04-28|        16|         1|
+--------+-----------+-------------+----------+----------+



In [10]:
supplier_summary = delayed_df.groupBy("supplier_id").count()
supplier_summary.show()

+-----------+-----+
|supplier_id|count|
+-----------+-----+
|        101|    1|
|        103|    1|
|        102|    1|
+-----------+-----+



In [11]:
supplier_summary.write.mode("overwrite").parquet("supplier_summary_parquet")

In [12]:
orders_df.createOrReplaceTempView("orders")

In [14]:
spark.sql("""SELECT supplier_id,COUNT(*) AS total_orders FROM orders
GROUP BY supplier_id""").show()

+-----------+------------+
|supplier_id|total_orders|
+-----------+------------+
|        101|           2|
|        103|           1|
|        102|           1|
|        104|           1|
+-----------+------------+

